# COMP9312 26T3 - Q1: earliest-connection queries

Implement `EarliestConnectionQuery.query(n, L, Q)`. Return one integer per query in order: earliest 1-based insertion time, `0` for identical endpoints, or `-1` if never connected.

Inputs are valid: `n >= 0`; IDs are `0..n-1`; no duplicate undirected edges or self-loops. Edge/query lists may be empty; query pairs may repeat. If `n == 0`, both lists are empty. Use only the Python Standard Library and keep public signatures unchanged. Put helpers/imports before the local-test cell. Read the accompanying Q1 specification and Project Summary.


In [1]:
class EarliestConnectionQuery:
    def query(self, n, L, Q):
        """Answer one batch using a merge forest and heavy-light LCA queries."""
        if not Q:
            return []

        # DSU roots are separate from history nodes: path compression must
        # never change the historical parent links used to answer queries.
        dsu = list(range(n))
        size = [1] * n
        top = list(range(n))

        def find(v):
            while dsu[v] != v:
                dsu[v] = dsu[dsu[v]]
                v = dsu[v]
            return v

        parent = list(range(n))
        left = [-1] * n
        right = [-1] * n
        time = [0] * n
        leaf_count = [1] * n

        for t, (u, v) in enumerate(L, 1):
            a, b = find(u), find(v)
            if a == b:
                continue  # This edge does not change connectivity.

            # A new parent represents the first union of these components.
            node = len(parent)
            x, y = top[a], top[b]
            parent.append(node)
            parent[x] = parent[y] = node
            left.append(x)
            right.append(y)
            time.append(t)
            leaf_count.append(size[a] + size[b])

            if size[a] < size[b]:
                a, b = b, a
            dsu[b] = a
            size[a] += size[b]
            top[a] = node

        # Heavy-light decomposition, using iterative DFS to handle histories
        # of depth n. Crossing a light edge at least halves the leaf count.
        depth = [0] * len(parent)
        head = [0] * len(parent)
        component = [0] * n
        for root, p in enumerate(parent):
            if p != root:
                continue
            stack = [(root, root)]
            while stack:
                node, chain = stack.pop()
                head[node] = chain
                if node < n:
                    component[node] = root
                    continue
                heavy, light = left[node], right[node]
                if leaf_count[heavy] < leaf_count[light]:
                    heavy, light = light, heavy
                depth[heavy] = depth[light] = depth[node] + 1
                stack.append((light, light))
                stack.append((heavy, chain))

        answers = []
        for u, v in Q:
            if u == v:
                answers.append(0)
            elif component[u] != component[v]:
                answers.append(-1)
            else:
                # Move the deeper chain head upward until both nodes lie
                # on one chain. The shallower node is then their LCA.
                while head[u] != head[v]:
                    if depth[head[u]] > depth[head[v]]:
                        u = parent[head[u]]
                    else:
                        v = parent[head[v]]
                ancestor = u if depth[u] < depth[v] else v
                answers.append(time[ancestor])
        return answers


## Algorithm and complexity

Process the insertion stream once with a disjoint-set union (DSU), using union by
size and path compression. Each vertex starts as a leaf. Whenever an edge joins
two different components, create a parent of their history roots and label it
with the edge's 1-based insertion time. Edges within a component need no history
node. There are at most `n - 1` unions, so the resulting binary forest has fewer
than `2n` nodes. DSU links and historical parent links are stored separately.

For different vertices in the same final component, their lowest common ancestor
(LCA) is precisely the union that first put them together. Before its labelled
time they are in separate components; at that time they become connected, and
later insertions cannot disconnect them. Thus its label is the required earliest
time. Different forest roots imply `-1`; identical endpoints give `0`.

A heavy-light decomposition supports LCA queries in linear auxiliary space.
Each internal node's child with more
leaves continues its heavy chain. A light child has at most half its parent's
leaves, so a root-to-leaf path crosses at most `floor(log2(n))` light edges.
Jump the deeper chain head upwards until both vertices are in the same chain;
the shallower remaining vertex is the LCA. All traversals are iterative, so even
a merge history of linear depth does not depend on Python's recursion limit.

Let `m = len(L)`, `q = len(Q)`, and let `alpha(n)` be the inverse Ackermann
function (take it as at least 1 for small `n`).

- **Stream processing:** `O(n + m alpha(n))`, including the `O(n)`
  initialisation even when many vertices are isolated. Each union adds `O(1)`
  historical data; building the heavy-light decomposition then costs `O(n)`.
- **One pair after preprocessing:** `O(log(n + 1))` worst case, and `O(1)`
  for identical or disconnected endpoints.
- **One complete batch:** `O(n + m alpha(n) + q log(n + 1))` time. The method
  preprocesses afresh on every call; this cost is not hidden in the per-pair
  bound. For `q > 0`, amortising the entire call gives
  `O((n + m alpha(n)) / q + log(n + 1))` per pair. An empty query batch returns
  immediately in `O(1)` time.
- **Auxiliary space:** `O(n)` for DSU, the forest, decomposition and iterative
  traversal stack. The input stream and query list occupy `O(m + q)` space
  separately, and the returned list uses `O(q)` space. Inputs are not modified.


## Local tests
Run after implementing. Do not change this cell. Timings are informational, not fixed grading limits. The test generator and full-output digests are deterministic.

In [2]:
# LOCAL TESTS - keep this marker and the tests unchanged.
"""Deterministic fixtures and independent oracles; Standard Library only."""
import hashlib
import heapq
from collections import deque


class FixedRandom:
    """Explicit xorshift32 test generator, independent of random module versions."""
    def __init__(self, seed):
        self.state = seed & 0xffffffff or 1

    def below(self, limit):
        x = self.state
        x ^= (x << 13) & 0xffffffff
        x ^= x >> 17
        x ^= (x << 5) & 0xffffffff
        self.state = x & 0xffffffff
        return self.state % limit


def graph(n, m, seed=1):
    assert 0 <= m <= n * (n - 1) // 2
    rng, seen, edges = FixedRandom(seed), set(), []
    while len(edges) < m:
        u, v = rng.below(n), rng.below(n)
        if u == v:
            continue
        edge = (min(u, v), max(u, v))
        if edge not in seen:
            seen.add(edge)
            edges.append(edge)  # Never enumerate the set.
    return edges


def pairs(n, count, seed=2):
    rng = FixedRandom(seed)
    return [(rng.below(n), rng.below(n)) for _ in range(count)]


def query_pairs(n, count, seed=3):
    rng = FixedRandom(seed)
    return [(rng.below(13), rng.below(n)) for _ in range(count)]


def add_digest(digest, values):
    digest.update((",".join(map(str, values)) + "\n").encode("ascii"))


def digest_rows(rows):
    digest = hashlib.sha256()
    for row in rows:
        add_digest(digest, row)
    return digest.hexdigest()


def brute_q1(n, edges, queries):
    answers = []
    for a, b in queries:
        if a == b:
            answers.append(0)
            continue
        adj, answer = [set() for _ in range(n)], -1
        for t, (u, v) in enumerate(edges, 1):
            adj[u].add(v)
            adj[v].add(u)
            seen, pending = {a}, [a]
            while pending:
                u = pending.pop()
                for v in adj[u] - seen:
                    seen.add(v)
                    pending.append(v)
            if b in seen:
                answer = t
                break
        answers.append(answer)
    return answers


def brute_q2(n, edges, k, v):
    adj = [set() for _ in range(n)]
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    alive, degree = [True] * n, [len(row) for row in adj]
    pending = deque(x for x in range(n) if degree[x] < k)
    while pending:
        x = pending.popleft()
        if not alive[x]:
            continue
        alive[x] = False
        for y in adj[x]:
            if alive[y]:
                degree[y] -= 1
                if degree[y] < k:
                    pending.append(y)
    if not alive[v]:
        return []
    seen, pending = {v}, [v]
    while pending:
        x = pending.pop()
        for y in adj[x]:
            if alive[y] and y not in seen:
                seen.add(y)
                pending.append(y)
    return sorted(seen)


def brute_max(n, edges, u, v):
    answer = -1
    for k in range(n + 1):
        if v in brute_q2(n, edges, k, u):
            answer = k
        else:
            break
    return answer



import time
sol = EarliestConnectionQuery()
assert sol.query(7, [(0,1),(2,3),(1,2),(4,5),(3,4),(0,3),(5,6)], [(0,4),(1,6),(0,2),(2,6),(5,3)]) == [5,7,3,7,5]
assert sol.query(5, [(0,1),(1,2),(3,4)], [(0,2),(2,3),(4,4)]) == [2,-1,0]
assert sol.query(0, [], []) == []
assert sol.query(1, [], [(0,0)]) == [0]
assert sol.query(5, [], [(0,4),(4,4),(0,4)]) == [-1,0,-1]
rng = FixedRandom(9312)
for trial in range(200):
    n = 2 + rng.below(11)
    E = graph(n, rng.below(min(20,n*(n-1)//2)+1), trial+10)
    Q = pairs(n,6,trial+500) + [(0,0)]
    assert sol.query(n,E,Q) == brute_q1(n,E,Q), (n,E,Q)
print("Q1 examples, boundary cases and 200 oracle trials passed")
n = 10000
E, Q = graph(n,80000,1), pairs(n,20000,2)
start = time.perf_counter()
answer = sol.query(n,E,Q)
elapsed = time.perf_counter()-start
assert type(answer) is list and len(answer)==len(Q)
assert all(type(value) is int for value in answer)
assert digest_rows([answer]) == '9da7db25f49df13f201b8d9077f4c60a77862e60306e31acf69af4e25e79d225', "Q1 full-output digest mismatch"
print("Q1 scale test passed; %.3f s (machine-dependent)" % elapsed)


Q1 examples, boundary cases and 200 oracle trials passed
Q1 scale test passed; 0.039 s (machine-dependent)
